# layoutpilot: research reproduction and extension

The code below is generated directly from the repository packages. Unit tests check implementation behavior, not reproduced paper benchmarks. Runs are illustrative and must be validated before claims of improvement.

In [ ]:
import torch
print("torch:",torch.__version__,"CUDA:",torch.cuda.is_available())
!pip -q install Pillow numpy pytest matplotlib opencv-python-headless

In [ ]:
!mkdir -p layoutpilot tests

In [ ]:
%%writefile layoutpilot/__init__.py
"""Layoutpilot research prototype package."""


In [ ]:
%%writefile layoutpilot/train.py
"""PPO from scratch for layout optimization; no Stable-Baselines or Gym dependency.
python train.py --steps 10000    # CPU-compatible
"""
import argparse,csv,random
from pathlib import Path
import numpy as np
import torch
from torch import nn
from torch.distributions import Categorical
from PIL import Image,ImageDraw,ImageFont

TARGET=np.array([[.08,.06,.84,.20],[.14,.32,.72,.43],[.16,.82,.68,.12]],dtype=np.float32)
# Three editable poster regions: headline, visual artwork, call to action.
# Their target geometry changes per brief; the RL policy must condition on it.
TEMPLATES={
    'minimal': TARGET,
    'editorial': np.array([[.08,.07,.84,.15],[.07,.30,.86,.49],[.55,.87,.38,.08]],dtype=np.float32),
    'split': np.array([[.08,.09,.39,.25],[.52,.09,.40,.69],[.08,.77,.37,.11]],dtype=np.float32),
}
N_ELEMS=3;N_ACTS=25;OBS_DIM=24

def observation(rects,target=TARGET):
    r=np.asarray(rects,dtype=np.float32).reshape(3,4)
    return np.concatenate([r.reshape(-1),((r-target)*4).reshape(-1)]).astype(np.float32)


def overlap(a,b):
    x0=max(float(a[0]),float(b[0]));y0=max(float(a[1]),float(b[1]))
    x1=min(float(a[0]+a[2]),float(b[0]+b[2]));y1=min(float(a[1]+a[3]),float(b[1]+b[3]))
    inter=max(x1-x0,0)*max(y1-y0,0)
    smaller=float(min(a[2]*a[3],b[2]*b[3]))
    # Occupancy of the smaller object: text legibility is lost even when
    # a small headline/CTA intersects a huge art block.
    return inter/max(smaller,1e-8)


def score(rects,target=TARGET):
    """Design proxy; higher is better, NOT human preference or learned aesthetics."""
    r=np.asarray(rects).reshape(3,4)
    centers=r[:,:2]+r[:,2:]/2; tc=target[:,:2]+target[:,2:]/2
    positioning=np.abs(centers-tc).mean()
    sizing=np.abs(r[:,2:]-target[:,2:]).mean()
    collisions=sum(overlap(r[i],r[j]) for i,j in [(0,1),(0,2),(1,2)])/3
    overflow=(np.maximum(-r[:,:2],0)+np.maximum(r[:,:2]+r[:,2:]-1,0)).mean()
    align=np.abs(centers[:,0]-tc[:,0]).mean()
    return float(1.-1.1*positioning-.7*sizing-.85*collisions-1.5*overflow-.18*align)


def clamp_to_artboard(r):
    """Hard safety constraints: all editable elements stay inside 4% margins."""
    r=np.asarray(r,dtype=np.float32).copy()
    r[:,2:]=np.clip(r[:,2:],.08,.86)
    r[:,0]=np.minimum(np.maximum(r[:,0],.04),.96-r[:,2])
    r[:,1]=np.minimum(np.maximum(r[:,1],.04),.96-r[:,3])
    return r


class PosterEnv:
    def __init__(self,seed=42,horizon=35,template='random'):
        if template!='random' and template not in TEMPLATES:raise ValueError('Unknown design template')
        self.rng=np.random.default_rng(seed);self.horizon=horizon;self.template=template;self.reset()
    def reset(self):
        self.template_name=(str(self.rng.choice(list(TEMPLATES))) if self.template=='random' else self.template)
        self.target=TEMPLATES[self.template_name]
        r=self.target+self.rng.normal(0,[.17,.17,.16,.10],(3,4)).astype(np.float32)
        r[:,:2]=np.clip(r[:,:2],-.05,.9);r[:,2:]=np.clip(r[:,2:],.10,.92)
        self.rects=clamp_to_artboard(r);self.t=0;return observation(self.rects,self.target)
    def step(self,action):
        old=score(self.rects,self.target)
        if int(action)!=24:
            item=int(action)//8;delta=int(action)%8
            axis=delta//2;sign=1 if delta%2==0 else -1
            val=.045 if axis<2 else .035
            self.rects[item,axis]+=sign*val
            if axis<2:self.rects[item,axis]=np.clip(self.rects[item,axis],-.1,.95)
            else:self.rects[item,axis]=np.clip(self.rects[item,axis],.08,.95)
        self.rects=clamp_to_artboard(self.rects)
        self.t+=1
        new=score(self.rects,self.target);reward=(new-old)*12-.003
        terminated=self.t>=self.horizon
        return observation(self.rects,self.target),float(reward),bool(terminated),{'quality':new}


class ActorCritic(nn.Module):
    def __init__(self,width=128,obs_dim=OBS_DIM,action_dim=N_ACTS):
        super().__init__()
        self.encoder=nn.Sequential(nn.Linear(obs_dim,width),nn.Tanh(),nn.Linear(width,width),nn.Tanh())
        self.actor=nn.Linear(width,action_dim);self.critic=nn.Linear(width,1)
        nn.init.orthogonal_(self.actor.weight,gain=.01)
        nn.init.orthogonal_(self.critic.weight,gain=1.)
    def forward(self,x):
        z=self.encoder(x);return self.actor(z),self.critic(z).squeeze(-1)


def rollout(env,model,device,count,gamma,lam):
    obs=observation(env.rects,env.target);arrays={k:[] for k in ('obs','actions','logp','values','rewards','dones')}
    for _ in range(count):
        x=torch.tensor(obs,device=device,dtype=torch.float32).unsqueeze(0)
        with torch.no_grad():
            logits,value=model(x);dist=Categorical(logits=logits);a=dist.sample()
        nxt,r,done,_=env.step(int(a.item()))
        for k,v in [('obs',obs.copy()),('actions',int(a.item())),('logp',float(dist.log_prob(a).item())),('values',float(value.item())),('rewards',r),('dones',float(done))]: arrays[k].append(v)
        obs=env.reset() if done else nxt
    with torch.no_grad():
        _,bootstrap=model(torch.tensor(obs,device=device,dtype=torch.float32).unsqueeze(0))
    values=np.asarray(arrays['values'],np.float32)
    adv=compute_gae(arrays['rewards'],values,arrays['dones'],float(bootstrap.item()),gamma,lam)
    arrays['returns']=adv+values;arrays['advantages']=adv
    return arrays


def clipped_policy_loss(new_log_prob,old_log_prob,advantages,clip=.2):
    """PPO-Clip: -E[min(r*A, clip(r,1-e,1+e)*A)] (Schulman et al.)."""
    ratio=(new_log_prob-old_log_prob).exp()
    return -torch.minimum(ratio*advantages,
        ratio.clamp(1-clip,1+clip)*advantages).mean()


def compute_gae(rewards,values,dones,last_value,gamma=.99,lam=.95):
    """GAE(lambda): terminal transitions MUST mask bootstrap and recursion."""
    n=len(rewards)
    advantages=np.zeros(n,dtype=np.float32)
    gae=0.
    for i in range(n-1,-1,-1):
        nonterminal=1.-float(dones[i])
        next_value=values[i+1] if i<n-1 else last_value
        delta=rewards[i]+gamma*next_value*nonterminal-values[i]
        gae=delta+gamma*lam*nonterminal*gae
        advantages[i]=gae
    return advantages


def ppo_update(model,opt,data,device,epochs=4,batch_size=128,clip=.2,entropy=.01,target_kl=None,return_diagnostics=False):
    obs=torch.tensor(np.asarray(data['obs']),dtype=torch.float32,device=device)
    actions=torch.tensor(data['actions'],dtype=torch.int64,device=device)
    oldlog=torch.tensor(data['logp'],dtype=torch.float32,device=device)
    returns=torch.tensor(data['returns'],dtype=torch.float32,device=device)
    advantages=torch.tensor(data['advantages'],dtype=torch.float32,device=device)
    advantages=(advantages-advantages.mean())/(advantages.std(unbiased=False)+1e-8)
    losses=[];kls=[];clip_fracs=[];stopped=False
    for epoch in range(epochs):
        for idx in torch.randperm(len(obs),device=device).split(batch_size):
            logits,v=model(obs[idx]);dist=Categorical(logits=logits)
            logratio=dist.log_prob(actions[idx])-oldlog[idx]
            ratio=logratio.exp()
            with torch.no_grad():
                approx_kl=((ratio-1)-logratio).mean().item()
                clip_fraction=((ratio-1).abs()>clip).float().mean().item()
            if target_kl is not None and approx_kl>1.5*target_kl:
                stopped=True;break
            policy_loss=clipped_policy_loss(dist.log_prob(actions[idx]),oldlog[idx],advantages[idx],clip)
            value_loss=(returns[idx]-v).square().mean()
            loss=policy_loss+.5*value_loss-entropy*dist.entropy().mean()
            opt.zero_grad();loss.backward();nn.utils.clip_grad_norm_(model.parameters(),.5);opt.step()
            losses.append(float(loss.detach()));kls.append(float(approx_kl));clip_fracs.append(float(clip_fraction))
        if stopped:break
    stats={'loss':float(np.mean(losses)) if losses else 0.0,
           'approx_kl':float(np.mean(kls)) if kls else 0.0,
           'clip_fraction':float(np.mean(clip_fracs)) if clip_fracs else 0.0,
           'early_stopped':stopped,'minibatches_updated':len(losses)}
    return stats if return_diagnostics else stats['loss']

def greedy_action(env):
    """Strong design-score oracle baseline; it directly inspects the reward."""
    prior=env.rects.copy();clock=env.t;best=24;best_score=score(prior,env.target)
    for a in range(N_ACTS):
        env.rects=prior.copy();env.t=clock
        _,_,_,info=env.step(a)
        if info['quality']>best_score+1e-8:best_score=info['quality'];best=a
    env.rects=prior;env.t=clock
    return best


def expert_warmstart(model,env,device,examples=0,epochs=6,batch_size=128,lr=1e-3):
    """Optional imitation from transparent reward-greedy demonstrations.

    This is BEHAVIORAL CLONING + PPO when enabled (not pure PPO). It can
    substantially accelerate learning in a tight Colab budget.
    """
    if examples<=0:return float('nan')
    xs=[];ys=[]
    # Sample fresh perturbed layouts; avoid a trivial dataset dominated by no-ops.
    attempts=0
    while len(xs)<examples and attempts<examples*20:
        attempts+=1
        env.reset()
        a=greedy_action(env)
        if a==24:continue
        xs.append(observation(env.rects,env.target).copy());ys.append(a)
    if len(xs)<examples:raise RuntimeError('Unable to collect diverse expert examples')
    features=torch.as_tensor(np.asarray(xs),dtype=torch.float32,device=device)
    actions=torch.as_tensor(ys,dtype=torch.long,device=device)
    opt=torch.optim.Adam(model.parameters(),lr=lr)
    losses=[]
    for _ in range(epochs):
        for ids in torch.randperm(examples,device=device).split(batch_size):
            logits,_=model(features[ids]);loss=nn.functional.cross_entropy(logits,actions[ids])
            opt.zero_grad();loss.backward();nn.utils.clip_grad_norm_(model.parameters(),1.);opt.step()
            losses.append(float(loss.detach()))
    return float(np.mean(losses[-max(1,examples//batch_size):]))


def evaluate(model,seeds,device,random_policy=False,horizon=35,greedy_policy=False):
    qualities=[];initial=[]
    for seed in seeds:
        env=PosterEnv(seed=seed,horizon=horizon);obs=env.reset()
        # save initial quality on exactly same independent seeds
        initial.append(score(env.rects,env.target))
        for j in range(env.horizon):
            if greedy_policy:a=greedy_action(env)
            elif random_policy: a=int(env.rng.integers(N_ACTS))
            else:
                with torch.no_grad():a=int(model(torch.tensor(obs,dtype=torch.float32,device=device).unsqueeze(0))[0].argmax(-1).item())
            obs,_,done,info=env.step(a)
            if done:break
        qualities.append(score(env.rects,env.target))
    return float(np.mean(initial)),float(np.mean(qualities))


def _font(sz,bold=False):
    # System fonts on Colab + Linux; independent of bundled proprietary fonts.
    paths=(['/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf'] if bold else
           ['/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf'])
    for path in paths:
        try:return ImageFont.truetype(path,max(8,int(sz)))
        except OSError:pass
    return ImageFont.load_default()


def render(rects,path,width=600,height=800,style='minimal'):
    """Poster PNG preview. Export separately to genuinely editable SVG."""
    palettes={
        'minimal':('#F4F1EB',['#14213D','#F1B85B','#187C7D'],'#182438'),
        'editorial':('#F5F0EA',['#D34D3F','#18324C','#E4AF58'],'#1F2D37'),
        'split':('#141720',['#F9C846','#4262D9','#D96350'],'#F6F2E8'),
    }
    bg,colors,ink=palettes.get(style,palettes['minimal'])
    im=Image.new('RGB',(width,height),bg);d=ImageDraw.Draw(im)
    d.text((42,9),'STUDIO  /  26     —     DESIGN EXPLORATION',fill=ink,font=_font(13,bold=True))
    # Draw artwork first; drawing text panels on top improves legibility under partial overlap.
    for i in [1,0,2]:
        x,y,w,h=rects[i]
        box=[int(x*width),int(y*height),int((x+w)*width),int((y+h)*height)]
        if box[2]<=box[0] or box[3]<=box[1]:continue
        d.rounded_rectangle(box,radius=16,fill=colors[i])
        bw=box[2]-box[0];bh=box[3]-box[1]
        if i==1:
            cx=(box[0]+box[2])/2;cy=(box[1]+box[3])/2
            rad=max(5,min(bw,bh)*.24)
            d.ellipse((cx-rad,cy-rad,cx+rad,cy+rad),fill='#F6EFE4')
            d.arc((cx-rad*.7,cy-rad*.7,cx+rad*.7,cy+rad*.7),45,290,fill='#D65350',width=6)
        else:
            label='CREATE / WITHOUT LIMITS' if i==0 else 'EXPLORE MORE  >'
            textcolor=('#FFFFFF' if i==0 else '#132438')
            availw=max(bw-24,10);availh=max(bh-14,10)
            size=min(34 if i==0 else 19, max(8,int(availh*.35)))
            while size>8 and d.textbbox((0,0),label,font=_font(size,bold=True))[2]>availw:
                size-=1
            d.text((box[0]+12,box[1]+max(5,(bh-size)//2-3)),label,
                   fill=textcolor,font=_font(size,bold=True))
    d.text((42,height-18),'CREATIVE LAB      /      REINFORCEMENT LEARNING',fill=ink,font=_font(11))
    im.save(path)


def export_svg(rects,path,style='minimal',width=600,height=800):
    """Native SVG primitives/text, not an embedded PNG; editable in Illustrator."""
    from xml.sax.saxutils import escape
    colors={
        'minimal':('#F4F1EB',['#14213D','#FCBD65','#187C7D']),
        'editorial':('#F5F0EA',['#D34D3F','#18324C','#E4AF58']),
        'split':('#141720',['#F9C846','#4262D9','#D96350']),
    }
    bg,palette=colors.get(style,colors['minimal'])
    lines=[f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
           f'<title>Editable {escape(style)} AI-designed poster</title>',
           f'<rect id="background" width="{width}" height="{height}" fill="{bg}"/>']
    labels=['CREATE WITHOUT LIMITS','VISUAL CANVAS','EXPLORE NOW']
    for i,(x,y,w,h) in enumerate(rects):
        x,y,w,h=(round(float(x*width),2),round(float(y*height),2),round(float(w*width),2),round(float(h*height),2))
        lines += [f'<g id="editable-layer-{i+1}" data-role="{["headline","artwork","cta"][i]}">',
                  f'<rect x="{x}" y="{y}" width="{w}" height="{h}" rx="12" fill="{palette[i]}"/>']
        if i==1:
            cx=x+w/2;cy=y+h/2;r=min(w,h)*.24
            lines.append(f'<circle cx="{cx:.2f}" cy="{cy:.2f}" r="{r:.2f}" fill="#F6EFE4"/>')
        else:
            textcolor='#132438' if i==2 else '#FFFFFF'
            lines.append(f'<text x="{x+12:.2f}" y="{y+28:.2f}" font-family="sans-serif" font-size="18" font-weight="bold" fill="{textcolor}">{escape(labels[i])}</text>')
        lines.append('</g>')
    lines.append('</svg>')
    Path(path).write_text('\n'.join(lines),encoding='utf-8')


def run(args):
    np.random.seed(args.seed);random.seed(args.seed);torch.manual_seed(args.seed)
    device=torch.device('cpu') # discrete lightweight simulator performs better on CPU
    out=Path(args.out);out.mkdir(parents=True,exist_ok=True)
    env=PosterEnv(seed=args.seed,horizon=args.horizon)
    model=ActorCritic(width=args.width).to(device)
    bc_loss=expert_warmstart(model,env,device,examples=args.warmstart_examples,epochs=args.warmstart_epochs)
    if args.warmstart_examples:
        print(f'Imitation warm-start used {args.warmstart_examples} reward-greedy examples, last BC loss {bc_loss:.4f}. Now PPO finetuning.')
        env.reset()
    opt=torch.optim.Adam(model.parameters(),lr=args.lr,eps=1e-5)
    hist=[];nupdates=max(1,(args.steps+args.rollout-1)//args.rollout)
    best_val=-float('inf');best_weights=None;best_update=0
    before_env=PosterEnv(seed=10001,horizon=args.horizon);initial_rects=before_env.rects.copy()
    for update in range(1,nupdates+1):
        batch=rollout(env,model,device,args.rollout,args.gamma,args.gae_lambda)
        diagnostics=ppo_update(model,opt,batch,device,epochs=args.epochs,batch_size=args.minibatch,
                               target_kl=args.target_kl if args.target_kl>0 else None,return_diagnostics=True)
        loss=diagnostics['loss']
        if update==1 or update==nupdates or update%max(1,nupdates//10)==0:
            q0,qa=evaluate(model,range(1000,1020),device,horizon=args.horizon)
            hist.append([update,update*args.rollout,loss,q0,qa,diagnostics['approx_kl'],diagnostics['clip_fraction'],int(diagnostics['early_stopped'])]);print(f"update {update}/{nupdates}: start={q0:.3f} PPO={qa:.3f} loss={loss:.3f} KL={diagnostics['approx_kl']:.4f}")
            # Select policy using validation seeds 1000..1019, never the final test seeds.
            if qa>best_val:
                best_val=qa;best_update=update
                best_weights={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    model.load_state_dict(best_weights)
    seeds=list(range(20000,20000+args.eval_episodes))
    qstart,qppo=evaluate(model,seeds,device,horizon=args.horizon)
    _,qrandom=evaluate(model,seeds,device,random_policy=True,horizon=args.horizon)
    _,qgreedy=evaluate(model,seeds,device,greedy_policy=True,horizon=args.horizon)
    demo=PosterEnv(seed=10001,horizon=args.horizon)
    obs=observation(demo.rects,demo.target)
    for _ in range(args.horizon):
        with torch.no_grad():action=int(model(torch.tensor(obs,dtype=torch.float32).unsqueeze(0))[0].argmax(-1).item())
        obs,*_=demo.step(action)
    render(initial_rects,out/'before.png',style=before_env.template_name)
    render(demo.rects,out/'after.png',style=demo.template_name)
    export_svg(demo.rects,out/'editable_poster.svg',style=demo.template_name)
    # Explicit geometric greedy oracle reference, NOT PPO output.
    oracle=PosterEnv(seed=10001,horizon=args.horizon)
    for _ in range(oracle.horizon):
        act=greedy_action(oracle);oracle.step(act)
    render(oracle.rects,out/'greedy_oracle.png',style=oracle.template_name)
    export_svg(oracle.rects,out/'greedy_oracle.svg',style=oracle.template_name)
    torch.save({'model':model.state_dict(),'width':args.width,'steps':args.steps,'selected_update':best_update},out/'policy.pt')
    with (out/'training.csv').open('w',newline='') as f:
        w=csv.writer(f);w.writerow(['update','steps','ppo_loss','mean_initial_quality','mean_ppo_quality','approx_kl','clip_fraction','kl_early_stop']);w.writerows(hist)
    import json
    metrics={'n_eval_episodes':args.eval_episodes,'eval_seed_start':20000,'validation_seed_start':1000,'target_kl':args.target_kl,'initial_score':qstart,'random_policy_score':qrandom,'greedy_oracle_score':qgreedy,'ppo_policy_score':qppo,'improvement_over_initial':qppo-qstart,'improvement_over_random':qppo-qrandom,'templates':list(TEMPLATES),'selected_validation_update':best_update,'best_validation_score':best_val,'expert_warmstart_examples':args.warmstart_examples,'bc_train_loss':bc_loss if args.warmstart_examples else None,'disclaimer':'Geometric proxy reward, NOT a validated human aesthetic measure.'}
    (out/'metrics.json').write_text(json.dumps(metrics,indent=2))
    print('Saved policy.pt, before.png, after.png, editable_poster.svg, greedy_oracle.png/svg, metrics.json, training.csv to',out)
    print(metrics)


def parse_args(argv=None):
    p=argparse.ArgumentParser();p.add_argument('--steps',type=int,default=16000)
    p.add_argument('--rollout',type=int,default=512);p.add_argument('--horizon',type=int,default=35)
    p.add_argument('--width',type=int,default=128);p.add_argument('--minibatch',type=int,default=128)
    p.add_argument('--epochs',type=int,default=4);p.add_argument('--gamma',type=float,default=.99)
    p.add_argument('--target-kl',type=float,default=.02,help='PPO stability guardrail; 0 disables KL early stopping')
    p.add_argument('--gae-lambda',type=float,default=.95);p.add_argument('--lr',type=float,default=3e-4)
    p.add_argument('--eval-episodes',type=int,default=60);p.add_argument('--seed',type=int,default=42)
    p.add_argument('--out',default='outputs')
    p.add_argument('--warmstart-examples',type=int,default=0,help='Optional greedy-expert BC examples; setting >0 makes this BC+PPO, not pure PPO')
    p.add_argument('--warmstart-epochs',type=int,default=6)
    return p.parse_args(argv)

if __name__=='__main__':run(parse_args())


In [ ]:
%%writefile layoutpilot/cartpole_sanity.py
"""Independent PPO-Clip on Gymnasium CartPole-v1 as a canonical RL sanity check.

This verifies generality of the algorithm outside the custom poster environment.
No Stable-Baselines implementation, pretraining, or RL helper library is used.
Optional dependency: pip install gymnasium
"""
import argparse, json
from pathlib import Path
import numpy as np
import torch
from torch.distributions import Categorical
from layoutpilot.train import ActorCritic, compute_gae, ppo_update


def train_cartpole(steps=8192,seed=42,out='cartpole_run',rollout_size=256,epochs=4):
    import gymnasium as gym
    np.random.seed(seed)
    torch.manual_seed(seed)
    env=gym.make('CartPole-v1')
    obs,_=env.reset(seed=seed)
    env.action_space.seed(seed)
    model=ActorCritic(width=64,obs_dim=4,action_dim=2)
    optimizer=torch.optim.Adam(model.parameters(),lr=2.5e-4,eps=1e-5)
    episode_rewards=[]
    this_return=0.
    for start in range(0,steps,rollout_size):
        count=min(steps-start,rollout_size)
        history={k:[] for k in ('obs','actions','logp','values','rewards','dones')}
        for _ in range(count):
            x=torch.tensor(obs,dtype=torch.float32).unsqueeze(0)
            with torch.no_grad():
                logits,v=model(x);dist=Categorical(logits=logits);a=dist.sample()
            obs_next,reward,terminated,truncated,_=env.step(int(a.item()))
            # A time-limit truncation is a bootstrap boundary, not an MDP terminal.
            # Bootstrap truncated terminal state into reward, then mask the GAE
            # recursion because the environment must reset to a new episode.
            adjusted_reward=float(reward)
            if truncated and not terminated:
                with torch.no_grad():
                    _,vnext=model(torch.tensor(obs_next,dtype=torch.float32).unsqueeze(0))
                adjusted_reward += .99*float(vnext.item())
            done=terminated or truncated
            for k,vv in [('obs',np.asarray(obs,dtype=np.float32).copy()),('actions',int(a.item())),
                ('logp',float(dist.log_prob(a).item())),('values',float(v.item())),
                ('rewards',adjusted_reward),('dones',float(done))]: history[k].append(vv)
            this_return+=float(reward)
            if done:
                episode_rewards.append(this_return)
                this_return=0.
                obs,_=env.reset()
            else:obs=obs_next
        with torch.no_grad():
            _,last_v=model(torch.tensor(obs,dtype=torch.float32).unsqueeze(0))
        v=np.asarray(history['values'],dtype=np.float32)
        advantage=compute_gae(history['rewards'],v,history['dones'],float(last_v.item()),.99,.95)
        history['advantages']=advantage;history['returns']=v+advantage
        ppo_update(model,optimizer,history,torch.device('cpu'),epochs=epochs,batch_size=64,clip=.2,entropy=.01)
    # Deterministic evaluation, independent of training random seeds.
    evals=[]
    for i in range(10):
        obs,_=env.reset(seed=seed+10_000+i)
        tot=0
        for _ in range(500):
            with torch.no_grad():
                logits,_=model(torch.tensor(obs,dtype=torch.float32).unsqueeze(0))
            obs,r,done,trunc,_=env.step(int(logits.argmax(-1)))
            tot+=float(r)
            if done or trunc:break
        evals.append(tot)
    env.close()
    path=Path(out);path.mkdir(parents=True,exist_ok=True)
    result={'total_steps':steps,'training_episodes':len(episode_rewards),
        'mean_recent_training_reward':float(np.mean(episode_rewards[-20:])) if episode_rewards else None,
        'mean_greedy_test_reward':float(np.mean(evals)),'test_rewards':evals,
        'note':'CartPole baseline sanity check, not an original PPO paper benchmark replication.'}
    (path/'cartpole_metrics.json').write_text(json.dumps(result,indent=2))
    torch.save({'model':model.state_dict(),'width':64,'obs_dim':4,'action_dim':2},path/'cartpole_policy.pt')
    print(json.dumps(result,indent=2))
    return result

if __name__=='__main__':
    p=argparse.ArgumentParser()
    p.add_argument('--steps',type=int,default=8192)
    p.add_argument('--seed',type=int,default=42)
    p.add_argument('--out',default='cartpole_run')
    args=p.parse_args()
    train_cartpole(args.steps,args.seed,args.out)


In [ ]:
%%writefile layoutpilot/evaluate_advanced.py
"""Paired fixed-seed policy comparison for LayoutPilot with bootstrap uncertainty.

Uses the identical starting layout for PPO-only and BC+PPO. Reports a handcrafted
geometric quality score, never calls this a user/aesthetic-preference benchmark.
"""
import argparse,csv,json
from pathlib import Path
import numpy as np
import torch
from layoutpilot.train import ActorCritic,PosterEnv,score,observation,greedy_action,N_ACTS


def bootstrap_ci(values,seed=2026,repeats=2000):
    data=np.asarray(values,dtype=float)
    if not len(data):raise ValueError('No values')
    rng=np.random.default_rng(seed)
    return [float(z) for z in np.quantile(data[rng.integers(0,len(data),(repeats,len(data)))].mean(axis=1),[.025,.975])]


def load_policy(path):
    checkpoint=torch.load(path,map_location='cpu',weights_only=True)
    model=ActorCritic(width=int(checkpoint['width'])).eval()
    model.load_state_dict(checkpoint['model'])
    return model


def rollout_score(model,seed,horizon=35,oracle=False,random=False):
    env=PosterEnv(seed=seed,horizon=horizon)
    obs=env.reset()
    initial=score(env.rects,env.target)
    for i in range(horizon):
        if oracle:a=greedy_action(env)
        elif random:a=int(env.rng.integers(N_ACTS))
        else:
            with torch.inference_mode():
                a=int(model(torch.from_numpy(obs).unsqueeze(0))[0].argmax(-1).item())
        obs,_,done,_=env.step(a)
        if done:break
    return initial,score(env.rects,env.target)


def compare(base,extended,n=40,seed_start=20000,horizon=35,out='paired_evaluation'):
    if n<=0:raise ValueError('n must be >0')
    base_model=load_policy(base);ext_model=load_policy(extended)
    rows=[]
    for s in range(seed_start,seed_start+n):
        ini,b=rollout_score(base_model,s,horizon)
        ini2,e=rollout_score(ext_model,s,horizon)
        _,r=rollout_score(None,s,horizon,random=True)
        _,g=rollout_score(None,s,horizon,oracle=True)
        assert abs(ini-ini2)<1e-7
        rows.append(dict(seed=s,initial=ini,random=r,ppo=b,bc_ppo=e,greedy=g,delta=e-b))
    result={'n':n,'test_seeds':[seed_start,seed_start+n-1],
            'means':{k:float(np.mean([row[k] for row in rows])) for k in ['initial','random','ppo','bc_ppo','greedy','delta']},
            'extension_minus_ppo_95pct_bootstrap_ci':bootstrap_ci([row['delta'] for row in rows]),
            'training_note':'PPO-only and BC+PPO checkpoints must use same budget and different output paths',
            'quality_note':'Synthetic handcrafted geometric score; not human preferences or industrial design assessment'}
    dest=Path(out);dest.mkdir(parents=True,exist_ok=True)
    with (dest/'paired_scores.csv').open('w',newline='') as f:
        writer=csv.DictWriter(f,fieldnames=list(rows[0]));writer.writeheader();writer.writerows(rows)
    (dest/'paired_summary.json').write_text(json.dumps(result,indent=2))
    print(json.dumps(result,indent=2))
    return result

if __name__=='__main__':
    p=argparse.ArgumentParser();p.add_argument('--ppo',required=True);p.add_argument('--bc-ppo',required=True)
    p.add_argument('--n',type=int,default=40);p.add_argument('--seed-start',type=int,default=20000)
    p.add_argument('--horizon',type=int,default=35);p.add_argument('--out',default='paired_evaluation')
    args=p.parse_args();compare(args.ppo,args.bc_ppo,args.n,args.seed_start,args.horizon,args.out)


In [ ]:
%%writefile layoutpilot/evaluate_reward_hacking.py
"""Small-scale, multi-seed PPO proxy-versus-constraint diagnostic.

PPO optimizes ONLY ``layoutpilot.train.score`` (proxy Q). A separate engineered
geometry metric measures approximate text occlusion, protected regions,
whitespace, and minimum text capacity. It is NOT a human preference oracle.

Unlike a step-zero random-policy comparison, all score deltas are paired against
untouched starting layouts. The x-axis is exact categorical KL from the frozen
initial policy on a fixed, held-out reference observation bank:
    E_s[ KL(pi_initial(.|s) || pi_current(.|s)) ].

No-op, random, and reward-aware greedy baselines use the SAME initial layouts.
Fixed reference states prevent changes in policy-induced occupancy from being
confounded with policy drift. KL measures policy change, not optimization quality.

Artifacts are ALWAYS labelled PIPELINE SMOKE TEST / exploratory; a longer run
alone cannot establish converged reward over-optimization or human preference.
Run from the repository root:
    python -m layoutpilot.evaluate_reward_hacking --steps 12288 \
        --rollout 256 --seeds 0 1 2 3 4
"""
from __future__ import annotations

import argparse
import csv
import json
import random
from pathlib import Path

import numpy as np
import torch

from layoutpilot.train import ActorCritic, PosterEnv, ppo_update, rollout, score, observation


def _intersection_fraction(a: np.ndarray, b: np.ndarray) -> float:
    """Area covered by b as a fraction of a; a and b are normalized xywh."""
    x0 = max(float(a[0]), float(b[0]))
    y0 = max(float(a[1]), float(b[1]))
    x1 = min(float(a[0] + a[2]), float(b[0] + b[2]))
    y1 = min(float(a[1] + a[3]), float(b[1] + b[3]))
    return float(max(0., x1-x0)*max(0., y1-y0)/max(1e-8,float(a[2]*a[3])))


def gold_metrics(rects: np.ndarray) -> dict[str, float]:
    """Independent, engineered constraint metrics; NOT measured human aesthetics.

    Inputs are element rectangles, not a proxy score or design target. A text
    ink-region approximation measures *hard* occlusion of headline and CTA by
    artwork/each other. A 16x16 occupancy grid measures whitespace imbalance.
    Additional reserved masthead and minimum text-capacity constraints are
    absent from the PPO training proxy Q. Some geometrical properties correlate
    with Q, so this is not an independent *human preference* oracle.
    """
    r = np.asarray(rects, dtype=np.float64).reshape(3, 4)
    headline = r[0]; artwork = r[1]; cta = r[2]
    # Approximate glyph-safe insets; no OCR or rendered-font dependency.
    def inset(rect, dx=.06, dy=.18):
        x,y,w,h = map(float, rect)
        return np.array([x+dx*w, y+dy*h, max(1e-6,w*(1-2*dx)),
                         max(1e-6,h*(1-2*dy))])
    ink_h, ink_c = inset(headline), inset(cta)
    hard_text_occlusion = 0.5*min(1., _intersection_fraction(ink_h, artwork) + _intersection_fraction(ink_h, cta)) + \
                          0.5*min(1., _intersection_fraction(ink_c, artwork) + _intersection_fraction(ink_c, headline))

    # Reserved masthead at top and legal/footer row at bottom; not rewarded by Q.
    masthead = np.array([.0,.0,1.0,.075])
    footer = np.array([.0,.925,1.0,.075])
    reserved_text_overlap = np.mean([max(_intersection_fraction(r[i],masthead),
                                          _intersection_fraction(r[i],footer))
                                      for i in (0,1,2)])

    # Deterministic occupancy of 16x16 grid; whitespace balance is deliberately
    # independent of target template geometry (unlike the proxy Q).
    coords = np.linspace(.04+(.92/32), .96-(.92/32), 16)
    yy,xx = np.meshgrid(coords,coords,indexing='ij')
    occupied = np.zeros((16,16), dtype=bool)
    for x,y,w,h in r:
        occupied |= (xx>=x)&(xx<x+w)&(yy>=y)&(yy<y+h)
    whitespace = ~occupied
    q = [whitespace[:8,:8].mean(),whitespace[:8,8:].mean(),
         whitespace[8:,:8].mean(),whitespace[8:,8:].mean()]
    whitespace_imbalance = float(np.clip(np.std(q)/.35,0,1))
    whitespace_shortfall = float(np.clip((.20-whitespace.mean())/.20,0,1))

    # Minimum dimensions approximate one readable headline and CTA line.
    text_capacity_penalty = float(np.clip(
        (max(0.,.42-headline[2])/.42 + max(0.,.12-headline[3])/.12 +
         max(0.,.26-cta[2])/.26 + max(0.,.07-cta[3])/.07)/4,0,1))
    gold = float(np.clip(1. - .40*hard_text_occlusion
                        - .20*reserved_text_overlap
                        - .20*whitespace_imbalance
                        - .10*whitespace_shortfall
                        - .10*text_capacity_penalty, 0.,1.))
    return {
        'gold_score': gold,
        'hard_text_occlusion': float(hard_text_occlusion),
        'reserved_text_overlap': float(reserved_text_overlap),
        'whitespace_imbalance': whitespace_imbalance,
        'whitespace_shortfall': whitespace_shortfall,
        'text_capacity_penalty': text_capacity_penalty,
    }



# Gold is intentionally target-blind. All baselines and policy rollouts use
# exactly the same deterministic initial rectangles for a given eval seed.
def evaluate_geometry(rects: np.ndarray, target: np.ndarray) -> dict[str, float]:
    """Score fixed geometry with both objectives; only proxy reads target."""
    return {'proxy_score': score(rects, target), **gold_metrics(rects)}


def _eval_episode(seed: int, horizon: int, policy: str,
                  model: ActorCritic | None = None) -> dict[str, float]:
    env = PosterEnv(seed=seed, horizon=horizon)
    start = evaluate_geometry(env.rects, env.target)
    if policy == 'random':
        # Uses a reproducible RNG independent of the policy and other seeds.
        action_rng = np.random.default_rng(seed + 1_000_000_007)
    for _ in range(horizon):
        if policy == 'noop':
            action = 24
        elif policy == 'random':
            action = int(action_rng.integers(0, 25))
        elif policy == 'greedy':
            from layoutpilot.train import greedy_action
            action = greedy_action(env)
        elif policy == 'model':
            if model is None:
                raise ValueError('model policy requires a model')
            obs = torch.as_tensor(observation(env.rects, env.target)).unsqueeze(0)
            with torch.no_grad():
                action = int(model(obs)[0].argmax(-1).item())
        else:
            raise ValueError(f'Unknown policy: {policy}')
        _, _, done, _ = env.step(action)
        if done:
            break
    end = evaluate_geometry(env.rects, env.target)
    result = {'eval_seed': seed, 'template': env.template_name,
              'initial_proxy': start['proxy_score'],
              'initial_gold': start['gold_score'], **end}
    result['proxy_delta'] = result['proxy_score'] - result['initial_proxy']
    result['gold_delta'] = result['gold_score'] - result['initial_gold']
    return result


def evaluate_policy(model: ActorCritic, eval_seeds: list[int], horizon: int):
    """Deterministic greedy-policy evaluation, paired to untouched starts."""
    was_training = model.training
    model.eval()
    try:
        return [_eval_episode(seed, horizon, 'model', model) for seed in eval_seeds]
    finally:
        model.train(was_training)


def evaluate_baselines(eval_seeds: list[int], horizon: int):
    """No-op/random/greedy, on the same held-out starting layouts."""
    return [{**{'baseline': policy}, **_eval_episode(seed, horizon, policy)}
            for policy in ('noop', 'random', 'greedy') for seed in eval_seeds]


def reference_observations(eval_seeds: list[int], horizon: int):
    """Fixed policy-independent states: starts and reproducible perturbations."""
    obs = []
    for seed in eval_seeds:
        env = PosterEnv(seed=seed, horizon=horizon)
        obs.append(observation(env.rects, env.target))
        action_rng = np.random.default_rng(seed + 2_000_000_011)
        for _ in range(min(horizon - 1, 8)):
            env.step(int(action_rng.integers(0, 25)))
            obs.append(observation(env.rects, env.target))
    return torch.from_numpy(np.stack(obs)).float()


def initial_policy_kl(model: ActorCritic, fixed_obs: torch.Tensor,
                      reference_logits: torch.Tensor) -> float:
    """Mean exact KL(pi_initial || pi_current) on identical fixed states."""
    from torch.distributions import Categorical, kl_divergence
    with torch.no_grad():
        current_logits, _ = model(fixed_obs)
        initial = Categorical(logits=reference_logits)
        current = Categorical(logits=current_logits)
        return float(kl_divergence(initial, current).mean().item())


def _mean_row(rows: list[dict], keys: tuple[str, ...]) -> dict[str, float]:
    return {key: float(np.mean([r[key] for r in rows])) for key in keys}


SCORE_KEYS = ('initial_proxy', 'initial_gold', 'proxy_score', 'gold_score',
              'proxy_delta', 'gold_delta', 'hard_text_occlusion',
              'reserved_text_overlap', 'whitespace_imbalance',
              'whitespace_shortfall', 'text_capacity_penalty')


def _write_csv(path: Path, data: list[dict]):
    if not data:
        raise ValueError(f'Cannot write empty CSV: {path}')
    with path.open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=list(data[0]))
        writer.writeheader()
        writer.writerows(data)


def plot_trajectories(trajectory: list[dict], baseline_rows: list[dict], out: Path):
    """KL is the x-axis, score deltas are anchored to starting geometry."""
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
    for ax, key, label in zip(axes, ('proxy_delta', 'gold_delta'),
                               ('Proxy Q − starting Q', 'Gold − starting gold')):
        for name, style in [('noop', ':'), ('random', '--'), ('greedy', '-.')]:
            value = float(np.mean([r[key] for r in baseline_rows if r['baseline'] == name]))
            ax.axhline(value, linestyle=style, linewidth=1.4,
                       label=f'{name}: {value:+.3f}')
        for seed in sorted({r['train_seed'] for r in trajectory}):
            points = sorted((r for r in trajectory if r['train_seed'] == seed),
                            key=lambda r: r['ppo_steps'])
            ax.plot([p['kl_from_initial'] for p in points],
                    [p[key] for p in points], marker='.', markersize=3,
                    linewidth=1, alpha=.75, label=f'PPO seed {seed}')
        ax.axvline(0, color='grey', linewidth=.6, alpha=.5)
        ax.set(xlabel=r'$E_{s\sim\mathcal{S}_{fixed}}[D_{KL}(\pi_0||\pi_\theta)]$',
               ylabel=label)
        ax.grid(alpha=.17)
    axes[0].legend(fontsize=7, ncol=2)
    axes[1].legend(fontsize=7, ncol=2)
    fig.suptitle('PIPELINE SMOKE TEST — PPO proxy vs engineered constraints\n'
                 'Fixed-state policy KL; no reward-hacking conclusion', fontsize=11)
    fig.tight_layout()
    fig.savefig(out, dpi=160)
    plt.close(fig)


def run(args):
    if any(getattr(args, k) < 1 for k in ('steps', 'rollout', 'horizon',
                                         'eval_episodes', 'epochs', 'minibatch', 'width')):
        raise ValueError('steps, rollout, horizon, eval-episodes, epochs, minibatch and width must be positive')
    if not args.seeds or len(args.seeds) != len(set(args.seeds)):
        raise ValueError('training seeds must be nonempty and unique')
    if not (0 < args.gamma <= 1 and 0 <= args.gae_lambda <= 1):
        raise ValueError('gamma and gae-lambda outside valid ranges')
    torch.set_num_threads(min(2, torch.get_num_threads()))
    output = Path(args.out)
    output.mkdir(parents=True, exist_ok=True)
    eval_seeds = list(range(args.eval_seed_start,
                            args.eval_seed_start + args.eval_episodes))
    reference_states = reference_observations(eval_seeds, args.horizon)
    baseline_rows = evaluate_baselines(eval_seeds, args.horizon)
    baseline_means = {name: _mean_row([r for r in baseline_rows if r['baseline'] == name],
                                      SCORE_KEYS)
                      for name in ('noop', 'random', 'greedy')}
    trajectory, episode_rows = [], []

    for seed in args.seeds:
        torch.manual_seed(seed)
        np.random.seed(seed)
        random.seed(seed)
        model = ActorCritic(width=args.width)
        model.eval()
        with torch.no_grad():
            initial_logits = model(reference_states)[0].detach().clone()
        optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, eps=1e-5)
        env = PosterEnv(seed=seed, horizon=args.horizon)
        checkpoints = list(range(0, args.steps, args.rollout)) + [args.steps]
        checkpoints = sorted(set(checkpoints))
        last_update = {'approx_kl': 0., 'clip_fraction': 0.,
                       'minibatches_updated': 0, 'early_stopped': False}
        for idx, step in enumerate(checkpoints):
            if idx:
                n_steps = step - checkpoints[idx - 1]
                batch = rollout(env, model, torch.device('cpu'), n_steps,
                                args.gamma, args.gae_lambda)
                last_update = ppo_update(model, optimizer, batch, torch.device('cpu'),
                                         epochs=args.epochs, batch_size=args.minibatch,
                                         clip=args.clip, target_kl=args.target_kl,
                                         return_diagnostics=True)
            evaluated = evaluate_policy(model, eval_seeds, args.horizon)
            for row in evaluated:
                episode_rows.append({'train_seed': seed, 'ppo_steps': step, **row})
            means = _mean_row(evaluated, SCORE_KEYS)
            entry = {'train_seed': seed, 'ppo_steps': step,
                     'kl_from_initial': initial_policy_kl(model, reference_states,
                                                           initial_logits),
                     'update_approx_kl': last_update['approx_kl'],
                     'update_clip_fraction': last_update['clip_fraction'],
                     'update_minibatches': last_update['minibatches_updated'],
                     'update_early_stopped': last_update['early_stopped'],
                     **means}
            trajectory.append(entry)
            print(f'seed={seed} steps={step} '
                  f'KL(init||now)={entry["kl_from_initial"]:.5f} '
                  f'Δproxy={entry["proxy_delta"]:+.3f} '
                  f'Δgold={entry["gold_delta"]:+.3f}', flush=True)

    _write_csv(output / 'reward_trajectories.csv', trajectory)
    _write_csv(output / 'episode_trajectories.csv', episode_rows)
    _write_csv(output / 'reference_baselines.csv', baseline_rows)
    plot_trajectories(trajectory, baseline_rows, output / 'reward_proxy_vs_gold.png')
    settings = vars(args).copy()
    settings.update({'torch_version': torch.__version__,
                     'eval_seeds': eval_seeds,
                     'reference_states': int(len(reference_states)),
                     'kl_definition': 'mean fixed-state categorical KL(initial || current)',
                     'baseline_definition': 'paired untreated initial layouts',
                     'gold_is_human_preference': False,
                     'experiment_status': 'PIPELINE_SMOKE_TEST',
                     'caveat': 'No claim of convergence, human preference, or reward hacking.'})
    (output / 'config.json').write_text(json.dumps(settings, indent=2) + '\n')
    per_seed = []
    for seed in args.seeds:
        rows = [r for r in trajectory if r['train_seed'] == seed]
        first, last = rows[0], rows[-1]
        per_seed.append({'train_seed': seed,
                         'initial_policy_kl': first['kl_from_initial'],
                         'final_policy_kl': last['kl_from_initial'],
                         'proxy_delta_vs_start': last['proxy_delta'],
                         'gold_delta_vs_start': last['gold_delta'],
                         'descriptive_proxy_gold_divergence':
                            last['proxy_delta'] > 0 and last['gold_delta'] < 0})
    (output / 'summary.json').write_text(json.dumps({
        'experiment_status': 'PIPELINE_SMOKE_TEST',
        'baselines_mean': baseline_means,
        'training_seeds': per_seed,
        'interpretation': 'Exploratory only: no causal, scaling-law, or convergence claim.'
    }, indent=2) + '\n')
    print('PIPELINE SMOKE TEST artifacts:', output.resolve())
    return trajectory


def parse_args(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--seeds', nargs='+', type=int, default=[0, 1, 2, 3, 4])
    parser.add_argument('--steps', type=int, default=12288,
                        help='PPO transitions per training seed; not a convergence guarantee')
    parser.add_argument('--rollout', type=int, default=256)
    parser.add_argument('--epochs', type=int, default=2)
    parser.add_argument('--minibatch', type=int, default=64)
    parser.add_argument('--width', type=int, default=64)
    parser.add_argument('--horizon', type=int, default=35)
    parser.add_argument('--eval-episodes', type=int, default=16)
    parser.add_argument('--eval-seed-start', type=int, default=10_000)
    parser.add_argument('--gamma', type=float, default=.99)
    parser.add_argument('--gae-lambda', type=float, default=.95)
    parser.add_argument('--lr', type=float, default=3e-4)
    parser.add_argument('--clip', type=float, default=.2)
    parser.add_argument('--target-kl', type=float, default=.02)
    parser.add_argument('--out', default='experiment_outputs/pipeline_smoke_reward_probe')
    return parser.parse_args(argv)


if __name__ == '__main__':
    run(parse_args())


In [ ]:
%%writefile layoutpilot/learned_proxy.py
"""Data, model and training primitives for learned-proxy reward experiments.

Gold labels are *only* used for supervised proxy training/validation, and
post-selection evaluation. Search is implemented in a separate module.
"""
from __future__ import annotations
from dataclasses import dataclass
import copy
import numpy as np
import torch
from torch import nn
from layoutpilot.train import TEMPLATES, clamp_to_artboard
from layoutpilot.evaluate_reward_hacking import gold_metrics


@dataclass(frozen=True)
class Template:
    uid: int
    target: np.ndarray
    family: str


def make_templates(count: int, seed_start: int) -> list[Template]:
    """Procedurally distinct target geometries; UID ranges must not overlap."""
    if count < 1:
        raise ValueError('count must be positive')
    bases = tuple(TEMPLATES.items())
    items = []
    for uid in range(seed_start, seed_start + count):
        rng = np.random.default_rng(uid)
        family, base = bases[int(rng.integers(len(bases)))]
        target = np.asarray(base, dtype=np.float32).copy()
        target += rng.normal(0, [.12, .12, .11, .07], (3, 4)).astype(np.float32)
        target = clamp_to_artboard(target)
        items.append(Template(uid, target, family))
    return items


def proposal_samples(target: np.ndarray, n: int, rng: np.random.Generator) -> np.ndarray:
    """Sample i.i.d. candidate layouts from a template-conditional proposal q0.

    Candidate rank is NOT influenced by gold scores. Sampling mixture includes
    out-of-distribution geometry to expose possible proxy extrapolation errors.
    """
    if n < 1:
        raise ValueError('n must be positive')
    target = np.asarray(target, dtype=np.float32).reshape(3, 4)
    noise = rng.normal(0, 1, (n, 3, 4)).astype(np.float32)
    scales = rng.choice(np.array([.35, 1.0, 2.1], dtype=np.float32), size=n,
                        p=[.3, .4, .3])
    sample = target[None, :, :] + noise * scales[:, None, None] * np.array(
        [.18, .18, .21, .12], dtype=np.float32)[None, None, :]
    # Twenty percent of the proposal is a broad independent mixture component.
    broad = rng.random(n) < .2
    if broad.any():
        k = int(broad.sum())
        sample[broad, :, :2] = rng.uniform(.02, .86, (k, 3, 2))
        sample[broad, :, 2:] = rng.uniform(.09, .7, (k, 3, 2))
    # clamp_to_artboard processes one layout, not a batch.
    return np.stack([clamp_to_artboard(r) for r in sample]).astype(np.float32)


def geometry_features(rects: np.ndarray) -> np.ndarray:
    """Target-blind 12-dimensional geometry features (no gold leakage)."""
    r = np.asarray(rects, dtype=np.float32)
    if r.shape[-2:] != (3, 4):
        raise ValueError('rects must end in (3,4)')
    if not np.all(np.isfinite(r)):
        raise ValueError('non-finite rectangle coordinates')
    return r.reshape(*r.shape[:-2], 12)


def labeled_samples(templates: list[Template], n: int, seed: int):
    """Gold evaluated for training LABELS, never during proxy optimization."""
    rng = np.random.default_rng(seed)
    group = rng.integers(0, len(templates), size=n)
    x = np.empty((n, 12), dtype=np.float32)
    y = np.empty(n, dtype=np.float32)
    for i, j in enumerate(group):
        rect = proposal_samples(templates[int(j)].target, 1, rng)[0]
        x[i] = geometry_features(rect)
        y[i] = gold_metrics(rect)['gold_score']
    return x, y


class ProxyMLP(nn.Module):
    def __init__(self, width: int = 64):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(12, width), nn.GELU(),
            nn.Linear(width, width), nn.GELU(), nn.Linear(width, 1),
            nn.Sigmoid())

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x).squeeze(-1)


def fit_proxy(train_x, train_y, val_x, val_y, *, seed: int,
              width: int = 64, updates: int = 250, batch_size: int = 128,
              lr: float = 0.001, device: str = 'cpu'):
    """Deterministic supervised training, validation-based checkpoint selection.

    Validation labels are used ONLY for selecting the checkpoint, not for search.
    """
    if updates < 1 or batch_size < 1 or width < 1:
        raise ValueError('updates, batch_size and width must be positive')
    torch.manual_seed(int(seed))
    dev = torch.device(device)
    model = ProxyMLP(width).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    tx = torch.from_numpy(np.asarray(train_x, dtype=np.float32)).to(dev)
    ty = torch.from_numpy(np.asarray(train_y, dtype=np.float32)).to(dev)
    vx = torch.from_numpy(np.asarray(val_x, dtype=np.float32)).to(dev)
    vy = torch.from_numpy(np.asarray(val_y, dtype=np.float32)).to(dev)
    gen = torch.Generator(device='cpu').manual_seed(seed + 821)
    best = float('inf')
    state = None
    best_step = 0
    log = []
    for step in range(1, updates + 1):
        idx = torch.randint(len(tx), (min(batch_size, len(tx)),), generator=gen).to(dev)
        model.train()
        loss = (model(tx[idx]) - ty[idx]).square().mean()
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        if step == 1 or step % 10 == 0 or step == updates:
            model.eval()
            with torch.no_grad():
                val_mse = float((model(vx) - vy).square().mean().item())
            log.append({'step': step, 'minibatch_mse': float(loss.item()), 'val_mse': val_mse})
            if val_mse < best:
                best = val_mse
                best_step = step
                state = copy.deepcopy(model.state_dict())
    assert state is not None
    model.load_state_dict(state)
    model.eval()
    return model, {'val_mse': best, 'selected_step': best_step, 'training_log': log}


def predict_proxy(model: ProxyMLP, features: np.ndarray, *, batch_size: int = 4096,
                  device: str = 'cpu') -> np.ndarray:
    """Batched forward pass: no calls to gold_metrics."""
    arr = np.asarray(features, dtype=np.float32).reshape(-1, 12)
    outputs = []
    dev = torch.device(device)
    model.eval()
    with torch.inference_mode():
        for start in range(0, len(arr), batch_size):
            x = torch.from_numpy(arr[start:start+batch_size]).to(dev)
            outputs.append(model(x).detach().cpu().numpy())
    return np.concatenate(outputs) if outputs else np.empty((0,), dtype=np.float32)


In [ ]:
%%writefile layoutpilot/evaluate_learned_proxy_hack.py
"""Learned reward proxy vs fixed gold: reproducible Best-of-N research probe.

Pipeline: train a proxy on gold-labeled layouts from TRAIN templates; checkpoint
on separate VALIDATION templates; sample IID proposals on 64+ HELD-OUT templates;
select using proxy ONLY; then evaluate gold on selected layouts. This does not
use PPO. The conditional proposal q0 and the induced BoN selection policy qN
have exactly known KLs when proxy scores have a continuous distribution:
  KL(qN || q0) = log(N) - (N-1)/N
  KL(q0 || qN) = (N-1) - log(N)
These measure KL between SELECTION DISTRIBUTIONS, not a PPO policy's KL.
The KL formula assumes IID candidates and no proxy-score ties (see README).

Run: python -m layoutpilot.evaluate_learned_proxy_hack --smoke
Full: python -m layoutpilot.evaluate_learned_proxy_hack --train-budgets 100 1000 10000
"""
from __future__ import annotations
import argparse
import csv
import json
import math
from pathlib import Path
import numpy as np
import torch
from layoutpilot.learned_proxy import (
    make_templates, proposal_samples, geometry_features,
    labeled_samples, fit_proxy, predict_proxy)
from layoutpilot.evaluate_reward_hacking import gold_metrics


def bon_kl(n: int) -> tuple[float, float]:
    """Exact forward/reverse conditional BoN-vs-proposal KL, in nats.

    For continuous proxy score F, dq_N/dq_0 = N*F(f(x))^(N-1).
    Therefore E_qN(log ratio)=log N-(N-1)/N,
    and E_q0(-log ratio)=(N-1)-log N.
    """
    if n < 1:
        raise ValueError('BoN candidate count must be positive')
    return math.log(n) - (n - 1) / n, (n - 1) - math.log(n)


def choose_proxy_only(scores: np.ndarray, budgets: list[int]) -> dict[int, int]:
    """Return candidate indices; cannot observe the true gold score by design."""
    a = np.asarray(scores, dtype=np.float64)
    if not np.all(np.isfinite(a)):
        raise ValueError('proxy scores must be finite')
    if not budgets or min(budgets) < 1 or max(budgets) > len(a):
        raise ValueError('budgets must lie within candidate bank')
    # Strict > tie handling picks the earliest best. Actual ties should be
    # checked: the analytic KL requires a continuous score distribution.
    indices = np.zeros(len(a), dtype=np.int64)
    current = 0
    for j in range(1, len(a)):
        if a[j] > a[current]:
            current = j
        indices[j] = current
    return {n: int(indices[n-1]) for n in budgets}


def evaluate_candidate_bank(model, templates, *, budgets, repeats, base_seed, device='cpu'):
    """A proxy-only search; gold is computed AFTER the choice has been frozen."""
    if repeats < 1 or len(templates) < 1:
        raise ValueError('nonempty eval templates/repeats required')
    rows = []
    biggest = max(budgets)
    for template in templates:
        for rep in range(repeats):
            rng = np.random.default_rng(base_seed + template.uid * 10_007 + rep * 1009)
            bank = proposal_samples(template.target, biggest, rng)
            proxy = predict_proxy(model, geometry_features(bank), device=device)
            chosen = choose_proxy_only(proxy, budgets)
            # All oracle calls occur AFTER proxy-only candidate selection.
            initial = gold_metrics(template.target)['gold_score']
            # Reference outcome is the first iid draw in the *same* proposal bank.
            # Compute it only after all candidate indices have been frozen.
            reference_gold = gold_metrics(bank[chosen[1]])['gold_score']
            reference_proxy = float(proxy[chosen[1]])
            for n in budgets:
                idx = chosen[n]
                metrics = gold_metrics(bank[idx])   # never used in choice
                kl_fwd, kl_rev = bon_kl(n)
                rows.append({
                    'template_uid': template.uid, 'template_family': template.family,
                    'repeat': rep, 'best_of_n': n,
                    'proxy_score': float(proxy[idx]),
                    'gold_score': metrics['gold_score'],
                    'gold_delta_from_start': metrics['gold_score'] - initial,
                    'gold_change_vs_n1': metrics['gold_score'] - reference_gold,
                    'proxy_change_vs_n1': float(proxy[idx]) - reference_proxy,
                    'starting_gold_score': initial,
                    'hard_text_occlusion': metrics['hard_text_occlusion'],
                    'whitespace_imbalance': metrics['whitespace_imbalance'],
                    'reserved_text_overlap': metrics['reserved_text_overlap'],
                    'text_capacity_penalty': metrics['text_capacity_penalty'],
                    'kl_selection_to_proposal': kl_fwd,
                    'kl_proposal_to_selection': kl_rev,
                    'selected_index': idx,
                })
    return rows


def hierarchical_bootstrap(rows, *, metrics, draws=500, seed=921):
    """95% paired hierarchical intervals, resampling TRAIN seeds & TEST templates.

    First aggregate repeats within a (proxy_seed, template_uid) pair; avoid
    treating repeated candidates from one template as independent test items.
    """
    if draws < 10:
        raise ValueError('bootstrap draws must be >= 10')
    rng = np.random.default_rng(seed)
    combos = sorted({(r['data_budget'], r['best_of_n']) for r in rows})
    output = []
    for budget, n in combos:
        selected = [r for r in rows if r['data_budget'] == budget and r['best_of_n'] == n]
        seeds = sorted({r['proxy_seed'] for r in selected})
        templates = sorted({r['template_uid'] for r in selected})
        index = {}
        for s in seeds:
            for t in templates:
                group = [r for r in selected if r['proxy_seed'] == s and r['template_uid'] == t]
                if not group:
                    raise ValueError('missing seed-template evaluation block')
                index[s, t] = {m: np.mean([r[m] for r in group]) for m in metrics}
        for metric in metrics:
            matrix = np.array([[index[s,t][metric] for t in templates] for s in seeds])
            samples = np.empty(draws)
            for i in range(draws):
                si = rng.integers(0, len(seeds), size=len(seeds))
                ti = rng.integers(0, len(templates), size=len(templates))
                samples[i] = float(matrix[np.ix_(si, ti)].mean())
            low, high = np.quantile(samples, [.025, .975])
            fwd, rev = bon_kl(n)
            output.append({'data_budget': budget, 'best_of_n': n,
                           'metric': metric, 'mean': float(matrix.mean()),
                           'ci95_low': float(low), 'ci95_high': float(high),
                           'n_proxy_seeds': len(seeds), 'n_test_templates': len(templates),
                           'kl_selection_to_proposal': fwd,
                           'kl_proposal_to_selection': rev})
    return output


def _csv(path, rows):
    if not rows:
        raise ValueError('empty CSV')
    with path.open('w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)


def _plot(summary, out, *, status):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
    metrics = [('proxy_score', 'Optimized learned proxy'),
               ('gold_score', 'Held-out fixed gold')]
    for ax, (metric, label) in zip(axes, metrics):
        for size in sorted({r['data_budget'] for r in summary}):
            points = sorted((r for r in summary if r['data_budget'] == size
                             and r['metric'] == metric), key=lambda r: r['best_of_n'])
            x = np.array([r['kl_selection_to_proposal'] for r in points])
            y = np.array([r['mean'] for r in points])
            lo = np.array([r['ci95_low'] for r in points])
            hi = np.array([r['ci95_high'] for r in points])
            ax.plot(x, y, marker='.', label=f'{size:,} gold labels')
            ax.fill_between(x, lo, hi, alpha=.12)
        ax.set(xlabel=r'$D_{KL}(q_N\|q_0)$ (nats; induced Best-of-N selection)',
               ylabel=label)
        ax.grid(alpha=.2)
    axes[1].legend(fontsize=8)
    fig.suptitle(f'{status} | Proxy-only Best-of-N, gold evaluated after selection',fontsize=10)
    fig.tight_layout()
    fig.savefig(out, dpi=150)
    plt.close(fig)


def run(args):
    budgets = sorted(set(args.train_budgets))
    bon = sorted(set(args.best_of_n))
    if min(budgets) < 2 or min(bon) < 1 or 1 not in bon:
        raise ValueError('positive training budgets and Best-of-N including N=1 required')
    if args.test_templates < 50:
        raise ValueError('at least 50 held-out templates required')
    if len(set(args.seeds)) != len(args.seeds) or not args.seeds:
        raise ValueError('proxy training seeds must be unique and nonempty')
    torch.set_num_threads(min(2, torch.get_num_threads()))
    train = make_templates(args.train_templates, 10_000)
    val = make_templates(args.val_templates, 200_000)
    test = make_templates(args.test_templates, 400_000)
    uid_sets = [set(x.uid for x in seq) for seq in [train, val, test]]
    assert not (uid_sets[0] & uid_sets[1] or uid_sets[0] & uid_sets[2] or uid_sets[1] & uid_sets[2])
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    status = 'PIPELINE_SMOKE_TEST' if args.smoke else 'UNVALIDATED_RESEARCH_RUN'
    rows, fit_info = [], []
    # A common candidate bank is used for each (template, repeat) across
    # training seeds and data budgets, reducing evaluation variance.
    for seed in args.seeds:
        vx, vy = labeled_samples(val, args.val_labels, seed=seed + 700_000)
        # Nested data regimes ensure that N=100 is a subset of N=1,000,
        # which in turn is a subset of N=10,000 for each training seed.
        all_tx, all_ty = labeled_samples(train, max(budgets), seed=seed + 60_000)
        for size in budgets:
            tx, ty = all_tx[:size], all_ty[:size]
            model, diagnostics = fit_proxy(tx, ty, vx, vy, seed=seed + 1_000,
                                           width=args.width, updates=args.updates,
                                           batch_size=args.batch_size, lr=args.lr,
                                           device=args.device)
            checkpoint_name = f'proxy_seed{seed}_labels{size}.pt'
            torch.save({'state_dict': model.state_dict(), 'width': args.width,
                        'seed': seed, 'n_train_labels': size,
                        'best_validation_step': diagnostics['selected_step']},
                       out / checkpoint_name)
            validation_predictions = predict_proxy(model, vx, device=args.device)
            corr = float(np.corrcoef(validation_predictions, vy)[0, 1]) if (
                np.std(validation_predictions) > 1e-12 and np.std(vy) > 1e-12) else float('nan')
            predicted = evaluate_candidate_bank(
                model, test, budgets=bon, repeats=args.repeats,
                base_seed=987_654, device=args.device)
            for r in predicted:
                r.update({'proxy_seed': seed, 'data_budget': size})
            rows.extend(predicted)
            fit_info.append({'proxy_seed': seed, 'data_budget': size,
                             'n_train_examples': len(tx),
                             'validation_mse': diagnostics['val_mse'],
                             'validation_mae': float(np.abs(validation_predictions - vy).mean()),
                             'validation_pearson_r': corr,
                             'best_training_step': diagnostics['selected_step'],
                             'checkpoint': checkpoint_name})
            print(f'proxy seed={seed} labels={size} '
                  f'val MSE={diagnostics["val_mse"]:.5f} '
                  f'BoN gold N={bon[-1]}: '
                  f'{np.mean([r["gold_score"] for r in predicted if r["best_of_n"]==bon[-1]]):.4f}',
                  flush=True)
    metrics = ['proxy_score', 'gold_score', 'gold_delta_from_start',
               'gold_change_vs_n1', 'proxy_change_vs_n1',
               'hard_text_occlusion', 'whitespace_imbalance']
    stats = hierarchical_bootstrap(rows, metrics=metrics, draws=args.bootstrap_draws)
    _csv(out / 'per_template.csv', rows)
    _csv(out / 'summary_ci.csv', stats)
    _csv(out / 'proxy_training.csv', fit_info)
    _plot(stats, out / 'proxy_gold_vs_kl.png', status=status)
    meta = {'experiment_status': status,
            'method': 'learned-proxy best-of-n selection; NOT PPO',
            'gold_is_human_preference': False,
            'gold_definition': 'target-blind engineered occlusion, whitespace and capacity',
            'KL_convention': 'qN=induced Best-of-N distribution, q0=proposal; both directions analytic in nats under IID continuous proxy scores',
            'CI_method': '95% hierarchical bootstrap over proxy seeds and held-out templates; repeats clustered within template',
            'train_template_uids': [x.uid for x in train],
            'validation_template_uids': [x.uid for x in val],
            'test_template_uids': [x.uid for x in test],
            'evaluation_gold_used_in_selection': False,
            'training_data_budget_design': 'nested labeled datasets within each proxy seed',
            'config': {k: v for k, v in vars(args).items() if k != 'out'},
            'caveat': 'A proxy/gold divergence is not guaranteed or established by a smoke test.'}
    (out / 'experiment.json').write_text(json.dumps(meta, indent=2) + '\n')
    return stats


def parse_args(argv=None):
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument('--smoke', action='store_true', help='tiny training, still >=50 held-out templates')
    p.add_argument('--train-budgets', type=int, nargs='+', default=[100, 1000, 10000])
    p.add_argument('--best-of-n', type=int, nargs='+', default=[1, 2, 4, 8, 16, 32, 64, 128, 256])
    p.add_argument('--seeds', type=int, nargs='+', default=[0, 1, 2])
    p.add_argument('--train-templates', type=int, default=32)
    p.add_argument('--val-templates', type=int, default=12)
    p.add_argument('--test-templates', type=int, default=64)
    p.add_argument('--val-labels', type=int, default=512)
    p.add_argument('--repeats', type=int, default=2)
    p.add_argument('--updates', type=int, default=300)
    p.add_argument('--width', type=int, default=64)
    p.add_argument('--batch-size', type=int, default=128)
    p.add_argument('--lr', type=float, default=.001)
    p.add_argument('--bootstrap-draws', type=int, default=500)
    p.add_argument('--device', choices=['cpu','cuda'], default='cpu')
    p.add_argument('--out', default='experiment_outputs/learned_proxy_study')
    args = p.parse_args(argv)
    if args.smoke:
        args.train_budgets = [100, 1000]
        args.best_of_n = [1, 2, 4, 8, 16]
        args.seeds = [0]
        args.updates = 20
        args.val_labels = 96
        args.repeats = 1
        args.bootstrap_draws = 100
    return args


if __name__ == '__main__':
    run(parse_args())


In [ ]:
%%writefile layoutpilot/demo.py
"""Interactive poster layout optimizer. Upload trained policy.pt and run `python demo.py`.
Self-contained demonstration; no external design software or proprietary SDK required.
"""
import argparse
import tempfile
from pathlib import Path
import numpy as np
import torch
from PIL import Image
from layoutpilot.train import PosterEnv,ActorCritic,observation,render,export_svg,TEMPLATES


def generate(policy,template='minimal',seed=42,horizon=35):
    env=PosterEnv(seed=int(seed),template=template,horizon=int(horizon))
    before=env.rects.copy();obs=observation(env.rects,env.target)
    for _ in range(env.horizon):
        with torch.inference_mode():
            logits,_=policy(torch.from_numpy(obs).unsqueeze(0))
            action=int(logits.argmax(-1).item())
        obs,_,done,_=env.step(action)
        if done:break
    out=Path(tempfile.mkdtemp(prefix='poster_'))
    render(before,out/'before.png',style=template)
    render(env.rects,out/'after.png',style=template)
    export_svg(env.rects,out/'editable.svg',style=template)
    return Image.open(out/'before.png'),Image.open(out/'after.png'),str(out/'editable.svg')


def create_app(checkpoint='outputs/policy.pt'):
    import gradio as gr
    state=torch.load(checkpoint,weights_only=True,map_location='cpu')
    model=ActorCritic(width=state.get('width',128))
    model.load_state_dict(state['model']);model.eval()
    return gr.Interface(lambda kind,seed:generate(model,kind,int(seed)),
        inputs=[gr.Dropdown(list(TEMPLATES),label='Design brief',value='editorial'),
                gr.Slider(0,10000,value=42,step=1,label='Layout seed')],
        outputs=[gr.Image(label='Before'),gr.Image(label='After'),gr.File(label='Editable layered SVG')],
        title='LayoutPilot · Poster Design Agent',
        description='PPO-trained geometric design optimizer. Templates/quality function are hand-defined, not human preference models. SVG output remains editable.')

if __name__=='__main__':
    p=argparse.ArgumentParser();p.add_argument('--checkpoint',default='outputs/policy.pt')
    create_app(p.parse_args().checkpoint).launch(share=False)


In [ ]:
%%writefile pytest.ini
[pytest]
testpaths = tests
pythonpath = .
addopts = --import-mode=importlib


In [ ]:
%%writefile tests/test_layoutpilot_advanced.py
import torch
import numpy as np
from layoutpilot.train import ActorCritic, PosterEnv,ppo_update,rollout
from layoutpilot.evaluate_advanced import rollout_score,compare

def test_ppo_kl_guardrail_and_observability():
    policy=ActorCritic(width=16);env=PosterEnv(seed=3,horizon=3)
    data=rollout(env,policy,'cpu',20,.99,.95)
    optimizer=torch.optim.Adam(policy.parameters(),lr=1e-4)
    d=ppo_update(policy,optimizer,data,'cpu',epochs=2,batch_size=8,target_kl=.02,return_diagnostics=True)
    assert d['minibatches_updated']>0
    assert np.isfinite(d['approx_kl']) and 0<=d['clip_fraction']<=1
    assert isinstance(d['early_stopped'],bool)

def test_identical_checkpoint_paired_score_zero(tmp_path):
    m=ActorCritic(width=16)
    ckpt=tmp_path/'policy.pt';torch.save({'model':m.state_dict(),'width':16},ckpt)
    result=compare(str(ckpt),str(ckpt),n=3,horizon=3,out=str(tmp_path/'eval'))
    assert abs(result['means']['delta'])<1e-9
    assert abs(result['extension_minus_ppo_95pct_bootstrap_ci'][0])<1e-9


In [ ]:
%%writefile tests/test_layoutpilot_learned_proxy.py
"""Failure-sensitive tests for the learned reward proxy study."""
import csv
import json
import math
import numpy as np
import pytest
from layoutpilot.learned_proxy import (
    make_templates, proposal_samples, geometry_features, labeled_samples,
    ProxyMLP, fit_proxy, predict_proxy)
from layoutpilot.evaluate_learned_proxy_hack import (
    bon_kl, choose_proxy_only, hierarchical_bootstrap, parse_args, run)


def test_exact_kl_conventions():
    assert bon_kl(1) == (0., 0.)
    for n in (2, 10, 100):
        f, r = bon_kl(n)
        assert f == pytest.approx(math.log(n) - (n-1)/n)
        assert r == pytest.approx(n-1-math.log(n))
        assert f > 0 and r > f
    with pytest.raises(ValueError):
        bon_kl(0)


def test_template_splits_procedural_and_disjoint():
    a = make_templates(12, 10_000)
    b = make_templates(12, 200_000)
    c = make_templates(64, 400_000)
    assert len({r.uid for r in a+b+c}) == 88
    assert len({tuple(r.target.ravel().round(5)) for r in c}) == 64


def test_proposal_and_features():
    t = make_templates(1, 10000)[0]
    x = proposal_samples(t.target, 20, np.random.default_rng(0))
    assert x.shape == (20, 3, 4)
    assert geometry_features(x).shape == (20, 12)
    assert np.isfinite(x).all()
    assert np.all(x[:,:,:2] >= .04-1e-5)
    with pytest.raises(ValueError):
        geometry_features(np.zeros((2,3)))


def test_proxy_selection_does_not_query_gold(monkeypatch):
    import layoutpilot.evaluate_learned_proxy_hack as exp
    def never_gold(*args, **kwargs):
        raise AssertionError('Selection must never call gold oracle')
    monkeypatch.setattr(exp, 'gold_metrics', never_gold)
    assert choose_proxy_only(np.array([.1,.8,.3,.7]), [1,2,3,4]) == {1:0, 2:1, 3:1, 4:1}
    assert choose_proxy_only(np.array([0.1,0.9,0.7]), [1,2,3])[3] == 1
    with pytest.raises(ValueError):
        choose_proxy_only(np.array([0.1,0.2]),[3])


def test_proxy_training_learns_and_validates():
    train = make_templates(3, 10_000)
    val = make_templates(3, 200_000)
    tx, ty = labeled_samples(train, 60, seed=7)
    vx, vy = labeled_samples(val, 24, seed=11)
    model, info = fit_proxy(tx, ty, vx, vy, seed=0, width=16, updates=12)
    ypred = predict_proxy(model, geometry_features(proposal_samples(
        train[0].target, 3, np.random.default_rng(2))))
    assert ypred.shape == (3,)
    assert np.all((ypred>=0)&(ypred<=1))
    assert math.isfinite(info['val_mse']) and info['selected_step'] >= 1


def test_hierarchical_ci_no_fake_independent_repeats():
    rows=[]
    for seed in (0,1):
        for template in (100,101,102):
            for repeat in (0,1):
                rows.append({'proxy_seed':seed,'template_uid':template,'data_budget':100,
                             'best_of_n':2,'repeat':repeat,'gold_score':.5+.1*seed})
    out = hierarchical_bootstrap(rows,metrics=['gold_score'], draws=50)
    assert len(out)==1 and out[0]['n_test_templates']==3
    assert out[0]['mean']==pytest.approx(.55)
    assert out[0]['ci95_low']<=out[0]['ci95_high']


def test_standalone_smoke_produces_heldout_and_two_kl_directions(tmp_path):
    args = parse_args(['--smoke', '--test-templates','50', '--out',str(tmp_path)])
    out = run(args)
    assert len(out)>0
    meta = json.loads((tmp_path/'experiment.json').read_text())
    assert meta['experiment_status']=='PIPELINE_SMOKE_TEST'
    assert len(meta['test_template_uids'])==50
    assert not set(meta['test_template_uids']) & set(meta['train_template_uids'])
    assert (tmp_path/'proxy_gold_vs_kl.png').stat().st_size > 1000
    assert (tmp_path/'proxy_seed0_labels100.pt').exists()
    with (tmp_path/'proxy_training.csv').open() as handle:
        train_rows = list(csv.DictReader(handle))
    assert {'validation_mae', 'validation_pearson_r', 'checkpoint'} <= set(train_rows[0])
    with (tmp_path/'per_template.csv').open() as f:
        rows=list(csv.DictReader(f))
    assert len(rows)==50*5*2
    assert all(float(r['kl_selection_to_proposal'])==pytest.approx(0.) for r in rows if r['best_of_n']=='1')
    assert all(float(r['kl_proposal_to_selection'])>0 for r in rows if int(r['best_of_n'])>1)


def test_selection_freezes_before_any_test_gold_query(monkeypatch):
    """Catch an implementation that quietly ranks candidates using the gold oracle."""
    import layoutpilot.evaluate_learned_proxy_hack as exp
    import torch
    from layoutpilot.evaluate_reward_hacking import gold_metrics as real_gold
    chosen = {'finished': False}
    real_choose = exp.choose_proxy_only

    def audited_select(scores, budgets):
        assert not chosen['finished']
        result = real_choose(scores, budgets)
        chosen['finished'] = True
        return result

    def gold_after_selection_only(rects):
        assert chosen['finished'], 'test gold queried before proxy selection finished'
        return real_gold(rects)

    monkeypatch.setattr(exp, 'choose_proxy_only', audited_select)
    monkeypatch.setattr(exp, 'gold_metrics', gold_after_selection_only)
    model = ProxyMLP(8)
    rows = exp.evaluate_candidate_bank(model, make_templates(1, 400000),
                                       budgets=[1, 2], repeats=1, base_seed=10)
    assert len(rows) == 2
    assert rows[0]['gold_change_vs_n1'] == pytest.approx(0.)
    assert rows[0]['proxy_change_vs_n1'] == pytest.approx(0.)


In [ ]:
%%writefile tests/test_layoutpilot_paper_fidelity.py
"""PPO mathematical objectives and GAE terminal masking."""
import numpy as np
import torch
from layoutpilot.train import clipped_policy_loss, compute_gae, PosterEnv, observation


def test_ratio_clipping_positive_and_negative_advantages():
    old=torch.zeros(2)
    new=torch.tensor([np.log(1.5), np.log(.5)],dtype=torch.float32)
    adv=torch.tensor([1.,-1.])
    # min(.5* -1, .8* -1)=-.8; min(1.5,1.2)=1.2 -> loss=-.2.
    expected=-(1.2-.8)/2
    assert torch.allclose(clipped_policy_loss(new,old,adv,clip=.2),torch.tensor(expected,dtype=torch.float32),atol=1e-6)


def test_gae_terminal_masks_future_rewards():
    rewards=[1.,7.,2.];values=[.5,.6,.7];dones=[0.,1.,0.]
    g=compute_gae(rewards,values,dones,last_value=123.,gamma=1.,lam=1.)
    # at terminal index 1: advantage=7-.6=6.4; index 0 includes only
    # 1 + gamma*6.4+(.6-.5) =7.5, not transition after terminal.
    assert abs(float(g[1])-6.4)<1e-5
    assert abs(float(g[0])-7.5)<1e-5
    assert abs(float(g[2])-(2+123-.7))<1e-5


def test_action_noop_and_observation_target_information():
    env=PosterEnv(seed=99,template='editorial',horizon=3)
    before=env.rects.copy()
    x=observation(env.rects,env.target)
    assert not np.isnan(x).any()
    env.step(24)
    assert np.array_equal(before,env.rects)
    assert np.all(env.rects[:,:2]+env.rects[:,2:]<=.96001)


def test_ppo_actor_critic_supports_standard_cartpole_shapes_and_gradients():
    from layoutpilot.train import ActorCritic, ppo_update
    model=ActorCritic(width=16,obs_dim=4,action_dim=2)
    optimizer=torch.optim.Adam(model.parameters(),lr=1e-3)
    obs=np.random.randn(16,4).astype('float32')
    with torch.no_grad():
        logits,v=model(torch.from_numpy(obs))
        actions=torch.distributions.Categorical(logits=logits).sample()
        lp=torch.distributions.Categorical(logits=logits).log_prob(actions)
    data={'obs':obs,'actions':actions.numpy(),'logp':lp.numpy(),
          'advantages':np.linspace(-1,1,16,dtype=np.float32),
          'returns':v.numpy() + .1}
    loss=ppo_update(model,optimizer,data,torch.device('cpu'),epochs=2,batch_size=8)
    assert np.isfinite(loss)
    assert model(torch.zeros(1,4))[0].shape==(1,2)


In [ ]:
%%writefile tests/test_layoutpilot_project.py
import torch
from layoutpilot.train import PosterEnv,ActorCritic,score,rollout,ppo_update

def test_environment():
    e=PosterEnv(seed=5,horizon=3);obs=e.reset();assert obs.shape==(24,)
    for i in range(3):
        obs,r,done,info=e.step(24)
    assert done and 'quality' in info and isinstance(score(e.rects),float)

def test_ppo_update():
    torch.manual_seed(1);env=PosterEnv(seed=2)
    model=ActorCritic(width=32);batch=rollout(env,model,torch.device('cpu'),32,.99,.95)
    assert len(batch['advantages'])==32
    optim=torch.optim.Adam(model.parameters(),lr=1e-3)
    before=model.actor.weight.detach().clone()
    loss=ppo_update(model,optim,batch,torch.device('cpu'),epochs=1,batch_size=16)
    assert torch.isfinite(torch.tensor(loss)) and not torch.equal(model.actor.weight,before)


def test_design_briefs_and_svg(tmp_path):
    from layoutpilot.train import TEMPLATES,export_svg,observation,greedy_action
    import xml.etree.ElementTree as ET
    for style in TEMPLATES:
        env=PosterEnv(seed=3,template=style)
        assert env.template_name==style
        assert observation(env.rects,env.target).shape==(24,)
        out=tmp_path/f'{style}.svg'
        export_svg(env.rects,out,style=style)
        root=ET.parse(out).getroot()
        assert root.tag.endswith('svg')
        assert sum(t.tag.endswith('g') for t in root)>=3
        assert 0<=greedy_action(env)<25


def test_optional_expert_warmstart():
    from layoutpilot.train import expert_warmstart
    model=ActorCritic(width=32)
    loss=expert_warmstart(model,PosterEnv(seed=7),torch.device('cpu'),examples=24,epochs=1,batch_size=16)
    assert torch.isfinite(torch.tensor(loss))


def test_hard_artboard_constraints():
    from layoutpilot.train import clamp_to_artboard
    import numpy as np
    boxes=np.array([[-.2,.9,1.2,.7],[.8,.8,.5,.5],[.9,-.5,.8,.3]])
    safe=clamp_to_artboard(boxes)
    assert np.all(safe[:,:2]>=.04-1e-6)
    assert np.all(safe[:,:2]+safe[:,2:]<=.96+1e-6)


In [ ]:
%%writefile tests/test_layoutpilot_reward_probe.py
"""Tests that discriminate target-blind gold constraints from target proxy Q."""
import numpy as np
import pytest
import torch
from layoutpilot.evaluate_reward_hacking import (
    _intersection_fraction,
    evaluate_baselines,
    evaluate_geometry,
    evaluate_policy,
    gold_metrics,
    initial_policy_kl,
    reference_observations,
)
from layoutpilot.train import ActorCritic


def test_gold_ignores_proxy_target():
    """Same geometry, different targets: proxy changes, independent gold cannot.

    Also construct a proxy-optimal but text-obscured layout: its Q improves
    when scored against its own bad target, but its gold score worsens.
    This would fail if gold accidentally read the target or mirrored Q.
    """
    clean = np.array([[.15, .14, .46, .16],
                      [.15, .40, .64, .30],
                      [.15, .83, .42, .09]], dtype=np.float32)
    occluded = clean.copy()
    occluded[1] = [.15, .15, .64, .55]  # artwork covers headline ink

    clean_target = clean.copy()
    bad_target = occluded.copy()
    a = evaluate_geometry(clean, clean_target)
    b = evaluate_geometry(clean, bad_target)
    assert a['proxy_score'] > b['proxy_score']  # target sensitivity
    assert a['gold_score'] == pytest.approx(b['gold_score'], abs=1e-12)
    assert a['hard_text_occlusion'] == b['hard_text_occlusion']

    # A safe but badly positioned/sized artwork block can score worse on Q
    # than a target-matching overlapping block, even though gold prefers it.
    safe_off_target = clean.copy()
    safe_off_target[1] = [.05, .75, .18, .15]
    proxy_good = evaluate_geometry(occluded, bad_target)
    proxy_bad = evaluate_geometry(safe_off_target, bad_target)
    assert proxy_good['proxy_score'] > proxy_bad['proxy_score']
    assert proxy_good['hard_text_occlusion'] > proxy_bad['hard_text_occlusion']
    assert proxy_good['gold_score'] < proxy_bad['gold_score']


def test_gold_penalizes_hard_occlusion():
    clean = np.array([[.15, .15, .45, .14], [.12, .36, .60, .32],
                      [.15, .82, .40, .10]])
    bad = clean.copy()
    bad[1] = [.16, .16, .50, .55]
    assert gold_metrics(bad)['hard_text_occlusion'] > gold_metrics(clean)['hard_text_occlusion']
    assert gold_metrics(bad)['gold_score'] < gold_metrics(clean)['gold_score']


def test_baselines_share_starts_and_noop_is_zero_delta():
    baselines = evaluate_baselines([10000, 10001], 4)
    assert len(baselines) == 6
    for seed in (10000, 10001):
        rows = [r for r in baselines if r['eval_seed'] == seed]
        assert len({r['initial_proxy'] for r in rows}) == 1
        assert len({r['initial_gold'] for r in rows}) == 1
        noop = next(r for r in rows if r['baseline'] == 'noop')
        assert noop['proxy_delta'] == pytest.approx(0., abs=1e-12)
        assert noop['gold_delta'] == pytest.approx(0., abs=1e-12)
    assert evaluate_baselines([10000, 10001], 4) == baselines


def test_probe_uses_fixed_eval_seeds_and_exposes_per_episode_data():
    model = ActorCritic(width=16)
    a = evaluate_policy(model, [10000, 10001], 3)
    b = evaluate_policy(model, [10000, 10001], 3)
    assert a == b and len(a) == 2
    assert all(0 <= e['gold_score'] <= 1 for e in a)
    assert all(e['proxy_delta'] == pytest.approx(e['proxy_score']-e['initial_proxy']) for e in a)


def test_kl_from_initial_zero_and_positive_after_policy_change():
    model = ActorCritic(width=16)
    states = reference_observations([10000, 10001], 5)
    assert states.shape[0] > 2
    with torch.no_grad():
        ref = model(states)[0].clone()
    assert initial_policy_kl(model, states, ref) == pytest.approx(0., abs=1e-6)
    with torch.no_grad():
        model.actor.bias[0].add_(2.5)
    assert initial_policy_kl(model, states, ref) > 0.01
    assert torch.equal(reference_observations([10000, 10001], 5), states)


def test_area_fraction():
    a = np.array([0, 0, 1, 1]); b = np.array([0, 0, .5, 1])
    assert abs(_intersection_fraction(a, b)-.5) < 1e-12


def test_probe_exports_labeled_baselines_kl_and_scores(tmp_path):
    """Exercise full CSV/PNG/metadata contract with two tiny PPO updates."""
    import csv
    import json
    from layoutpilot.evaluate_reward_hacking import parse_args, run
    args = parse_args(['--seeds', '0', '1', '--steps', '32', '--rollout', '16',
                       '--epochs', '1', '--width', '16', '--horizon', '3',
                       '--eval-episodes', '2', '--out', str(tmp_path)])
    rows = run(args)
    assert len(rows) == 6  # 2 training seeds * (step 0, 16, 32)
    assert all(r['kl_from_initial'] < 1e-6 for r in rows if r['ppo_steps'] == 0)
    with (tmp_path / 'reference_baselines.csv').open() as handle:
        baselines = list(csv.DictReader(handle))
    assert {r['baseline'] for r in baselines} == {'noop', 'random', 'greedy'}
    assert all(float(r['proxy_delta']) == pytest.approx(0., abs=1e-12)
               for r in baselines if r['baseline'] == 'noop')
    assert (tmp_path / 'reward_proxy_vs_gold.png').stat().st_size > 1000
    assert json.loads((tmp_path / 'summary.json').read_text())['experiment_status'] == 'PIPELINE_SMOKE_TEST'
    assert json.loads((tmp_path / 'config.json').read_text())['gold_is_human_preference'] is False


In [ ]:
!python -m pytest -q tests/test_layoutpilot_*.py

## Controlled experiments
Each training configuration is a starting point, not evidence of convergence.

In [ ]:
!python -m layoutpilot.train --steps 6000 --rollout 256 --warmstart-examples 0 --seed 42 --out baseline

In [ ]:
!python -m layoutpilot.train --steps 6000 --rollout 256 --warmstart-examples 700 --seed 42 --out extension

In [ ]:
!python -m layoutpilot.evaluate_advanced --ppo baseline/policy.pt --bc-ppo extension/policy.pt --n 40 --out paired_eval

In [ ]:
!python -m layoutpilot.evaluate_reward_hacking --steps 12288 --rollout 256 --seeds 0 1 2 3 4 --out pipeline_smoke_reward_probe

In [ ]:
!python -m layoutpilot.evaluate_learned_proxy_hack --smoke --out learned_proxy_smoke

In [ ]:
from IPython.display import display
from PIL import Image
display(Image.open("pipeline_smoke_reward_probe/reward_proxy_vs_gold.png"))
display(Image.open("learned_proxy_smoke/proxy_gold_vs_kl.png"))

## Preserve artifacts
Download before the Colab runtime is recycled.

In [ ]:
from pathlib import Path
from google.colab import files
import shutil
for folder in ('baseline','extension','paired_eval','pipeline_smoke_reward_probe','learned_proxy_smoke','eval_baseline','eval_extension'):
    if Path(folder).is_dir():
        archive=shutil.make_archive(folder,'zip',folder)
        files.download(archive)